In [20]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt

In [21]:
data_path = "../data/datasets/panel_dataset_VIF_normalized.xlsx"
df = pd.read_excel(data_path)
all_results = []
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [55]:
results = []

In [161]:
TARGETS = ['RECESS', 'RECESS_OVER', 'RECESS_PERIOD']
target_name = "RECESS"
country = "JAP"
time_steps = 12
group = df[df["Country"] == country].copy().reset_index(drop=True)

In [162]:
drop_cols = ['date', 'Country', 'month'] + TARGETS
feature_cols = [col for col in group.columns if col not in drop_cols]
group.dropna(subset=feature_cols + [target_name], inplace=True)

In [163]:
print("Number of features:", len(feature_cols))
print("First 5 features:", feature_cols[:5])

Number of features: 19
First 5 features: ['CCI (Index)_LAG12', 'CCI (Index)_LAG3', 'EPU (Index)_LAG1', 'EPU (Index)_LAG12', 'EPU (Index)_LAG3']


In [164]:
train_start = "1995-01-01"
train_end = "2009-12-31"
val_end = "2018-12-31"
test_end = "2025-05-31"

In [165]:
print(group[feature_cols].isnull().sum().sort_values(ascending=False).head(10))

CCI (Index)_LAG12    0
CCI (Index)_LAG3     0
EPU (Index)_LAG1     0
EPU (Index)_LAG12    0
EPU (Index)_LAG3     0
EPU (Index)_LAG6     0
INF (%)              0
INF (%)_LAG1         0
INF (%)_LAG12        0
INF (%)_LAG3         0
dtype: int64


In [166]:
X_seq, y_seq, date_seq = [], [], []
for i in range(len(group) - time_steps):
    X_seq.append(group.iloc[i:i+time_steps, :][feature_cols].values)
    y_seq.append(group.loc[i + time_steps, target_name])
    date_seq.append(group.loc[i + time_steps, 'date'])

X = np.array(X_seq)
y = np.array(y_seq)
dates = pd.to_datetime(date_seq)

In [167]:
train_mask = (dates >= train_start) & (dates <= train_end)
val_mask   = (dates > train_end) & (dates <= val_end)
test_mask  = (dates > val_end) & (dates <= test_end)

In [168]:
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val     = X[val_mask], y[val_mask]
X_test, y_test   = X[test_mask], y[test_mask]

In [169]:
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

Train: 168 samples
Validation: 108 samples
Test: 64 samples


In [170]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

In [171]:
def build_standard_lstm(hp):
    model = Sequential()
    model.add(LSTM(
        units=hp.Choice('units', [64, 128, 256]),
        input_shape=(X_train.shape[1], X_train.shape[2]),
        return_sequences=False,
        recurrent_dropout=hp.Float('rec_dropout', 0.0, 0.3, step=0.1)
    ))
    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [172]:
# Bidirectional LSTM
def build_bi_lstm(hp):
    model = Sequential()
    model.add(Bidirectional(LSTM(
        units=hp.Choice('units', [64, 128, 256]),
        recurrent_dropout=hp.Float('rec_dropout', 0.0, 0.3, step=0.1)
    ), input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [173]:
# Stacked LSTM
def build_stacked_lstm(hp):
    model = Sequential()
    model.add(LSTM(
        units=hp.Choice('units_1', [64, 128, 256]),
        return_sequences=True,
        input_shape=(X_train.shape[1], X_train.shape[2]),
        recurrent_dropout=hp.Float('rec_dropout_1', 0.0, 0.3, step=0.1)
    ))
    model.add(Dropout(hp.Float('dropout_1', 0.2, 0.5, step=0.1)))
    model.add(LSTM(
        units=hp.Choice('units_2', [32, 64]),
        return_sequences=False,
        recurrent_dropout=hp.Float('rec_dropout_2', 0.0, 0.3, step=0.1)
    ))
    model.add(Dropout(hp.Float('dropout_2', 0.2, 0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [174]:
def run_tuner(build_model_fn, project_name):
    tuner = kt.RandomSearch(
        build_model_fn,
        objective='val_accuracy',
        max_trials=10,
        executions_per_trial=1,
        directory='tuner_logs',
        project_name=project_name
    )

    tuner.search(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=[early_stop],
        verbose=1
    )

    best_model = tuner.get_best_models(1)[0]
    best_hps = tuner.get_best_hyperparameters(1)[0]

    y_pred_prob = best_model.predict(X_test).flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)

    return {
        "Country": country,
        "Architecture": project_name.replace("tune_", "").replace("_JAP", "").replace("_RECESS", ""),
        "Target": target_name,
        "AUC": roc_auc_score(y_test, y_pred_prob),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Threshold": 0.5,
        "Best_Hyperparams": best_hps.values
    }


In [175]:
results.append(run_tuner(build_standard_lstm, f"tune_standard_{country}_RECESS"))

Trial 10 Complete [00h 00m 11s]
val_accuracy: 0.6944444179534912

Best val_accuracy So Far: 0.8055555820465088
Total elapsed time: 00h 01m 40s


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 7 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step 


In [176]:
results.append(run_tuner(build_bi_lstm, f"tune_bidirectional_{country}_RECESS"))

Trial 10 Complete [00h 00m 13s]
val_accuracy: 0.6296296119689941

Best val_accuracy So Far: 0.7685185074806213
Total elapsed time: 00h 01m 58s


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step 


In [177]:
results.append(run_tuner(build_stacked_lstm, f"tune_stacked_{country}_RECESS"))

Trial 10 Complete [00h 00m 19s]
val_accuracy: 0.5925925970077515

Best val_accuracy So Far: 0.8055555820465088
Total elapsed time: 00h 02m 48s


C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step 


In [178]:
results_df = pd.DataFrame(results)
results_df

,Country,Architecture,Target,AUC,F1,Precision,Recall,Accuracy,Threshold,Best_Hyperparams
0,CAN,standard_CAN,RECESS,0.704082,0.372093,0.275862,0.571429,0.571429,0.5,"{'units': 64, 'rec_dropout': 0.2, 'dropout': 0..."
1,CAN,bidirectional_CAN,RECESS,0.623907,0.405797,0.254545,1.000000,0.349206,0.5,"{'units': 256, 'rec_dropout': 0.1, 'dropout': ..."
2,CAN,stacked_CAN,RECESS,0.663265,0.417910,0.264151,1.000000,0.380952,0.5,"{'units_1': 128, 'rec_dropout_1': 0.0, 'dropou..."
3,USA,standard_USA,RECESS,0.984127,0.067797,0.035088,1.000000,0.153846,0.5,"{'units': 64, 'rec_dropout': 0.2, 'dropout': 0..."
4,USA,bidirectional_USA,RECESS,0.976190,0.085106,0.044444,1.000000,0.338462,0.5,"{'units': 256, 'rec_dropout': 0.1, 'dropout': ..."
5,USA,stacked_USA,RECESS,0.984127,0.064516,0.033333,1.000000,0.107692,0.5,"{'units_1': 256, 'rec_dropout_1': 0.2, 'dropou..."
6,MEX,standard_MEX,RECESS,0.960317,0.075472,0.039216,1.000000,0.246154,0.5,"{'units': 128, 'rec_dropout': 0.1, 'dropout': ..."
7,MEX,bidirectional_MEX,RECESS,0.976190,0.075472,0.039216,1.000000,0.246154,0.5,"{'units': 256, 'rec_dropout': 0.2, 'dropout': ..."
8,MEX,stacked_MEX,RECESS,0.976190,0.078431,0.040816,1.000000,0.276923,0.5,"{'units_1': 64, 'rec_dropout_1': 0.1, 'dropout..."
9,GER,standard_GER,RECESS,1.000000,0.944444,0.894737,1.000000,0.968750,0.5,"{'units': 64, 'rec_dropout': 0.1, 'dropout': 0..."
